# MusselApp · Entrenar el detector de choritos y cholgas (RF-DETR Nano)

Notebook para **Google Colab gratis** (GPU T4). Usa **RF-DETR Nano** (licencia Apache 2.0) y guarda todo en el
Drive del equipo (`MusselApp-ML/`), así que si Colab corta la sesión, se reanuda donde quedó.

**Antes de correrlo:** Entorno de ejecución → Cambiar tipo de entorno → **GPU T4**.

Pasos:
1. Preparar el entorno y montar Drive.
2. Unir las etiquetas de Label Studio y separar por bandeja (sin fuga de datos).
3. (Opcional) Pre-entrenar con un dataset público.
4. Entrenar con nuestras fotos (se reanuda solo).
5. Medir contra las metas del go/no-go.
6. Exportar a `.tflite` para la app (en un entorno nuevo; ver la sección 6).

## 1. Entorno y Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

# Carpeta compartida por Lucas. Si no aparece en "Mi unidad", agrega un acceso directo a ella desde "Compartido conmigo".
DRIVE = '/content/drive/MyDrive/MusselApp-ML'
import os
assert os.path.isdir(DRIVE), f'No encuentro {DRIVE}: agrega el acceso directo a la carpeta MusselApp-ML en tu Drive'
print(os.listdir(DRIVE))

In [ ]:
# Versión fija (probada con la documentación del 6-nov-2026). No cambiarla sin volver a probar todo.
!pip install -q "rfdetr[train]==1.12.0"
!git clone -q https://github.com/lucsaldivia-ux/MusselApp-ML.git /content/MusselApp-ML || (cd /content/MusselApp-ML && git pull -q)

## 2. Unir las etiquetas y separar por bandeja

Cada integrante sube a `dataset_v1/etiquetas/` el export **COCO** de Label Studio (un `.zip` con `result.json` e
`images/`). Aquí se descomprimen, se unen y se separan en entrenamiento / validación / prueba **por bandeja**:
las fotos de una misma bandeja nunca quedan en conjuntos distintos.

In [ ]:
import glob, json, shutil, zipfile
from pathlib import Path

TRABAJO = Path('/content/trabajo')
shutil.rmtree(TRABAJO, ignore_errors=True)
EXTRAIDO = TRABAJO / 'extraido'
FOTOS = TRABAJO / 'fotos'
FOTOS.mkdir(parents=True)

jsons = []
for i, z in enumerate(sorted(glob.glob(f'{DRIVE}/dataset_v1/etiquetas/*.zip'))):
    destino = EXTRAIDO / f'lote{i}'
    zipfile.ZipFile(z).extractall(destino)
    jsons += glob.glob(str(destino / '**' / '*.json'), recursive=True)
    for img in glob.glob(str(destino / '**' / '*.*'), recursive=True):
        if img.lower().endswith(('.jpg', '.jpeg', '.png')):
            shutil.copy(img, FOTOS / Path(img).name)
print(f'{len(jsons)} archivos de etiquetas, {len(list(FOTOS.iterdir()))} fotos')
assert jsons, 'No hay exports en dataset_v1/etiquetas/'

In [ ]:
!python /content/MusselApp-ML/src/split_sin_fuga.py {' '.join(jsons)} --salida /content/trabajo/split --semilla 42

In [ ]:
# Carpeta en el formato que espera RF-DETR: train/, valid/, test/ con _annotations.coco.json y las fotos
DATASET = TRABAJO / 'dataset'
for origen, destino in [('train', 'train'), ('val', 'valid'), ('test', 'test')]:
    coco = json.load(open(TRABAJO / 'split' / f'{origen}.json'))
    carpeta = DATASET / destino
    carpeta.mkdir(parents=True)
    faltan = 0
    for im in coco['images']:
        archivo = FOTOS / im['file_name']
        if archivo.exists():
            shutil.copy(archivo, carpeta / im['file_name'])
        else:
            faltan += 1
    json.dump(coco, open(carpeta / '_annotations.coco.json', 'w'))
    print(f"{destino}: {len(coco['images'])} fotos, {len(coco['annotations'])} etiquetas, {faltan} fotos sin archivo")
CLASES = [c['name'] for c in coco['categories']]
print('Clases:', CLASES)

## 3. (Opcional) Pre-entrenar con un dataset público

Sirve para que el modelo aprenda antes a encontrar "un individuo". Solo datasets **CC BY 4.0 / CC0** listados en
`DATOS_EXTERNOS.md`. Descarga el dataset desde Roboflow Universe en formato **COCO** (cuenta gratis), súbelo a
`MusselApp-ML/externos/` como `.zip` y pon `USAR_EXTERNO = True`. Si no, salta esta sección.

In [ ]:
USAR_EXTERNO = False
PREENTRENO = f'{DRIVE}/checkpoints/preentreno_externo'

if USAR_EXTERNO:
    from rfdetr import RFDETRNano
    externo = TRABAJO / 'externo'
    zipfile.ZipFile(glob.glob(f'{DRIVE}/externos/*.zip')[0]).extractall(externo)
    modelo = RFDETRNano()
    modelo.train(dataset_dir=str(externo), epochs=15, batch_size='auto', lr=1e-4, output_dir=PREENTRENO,
                 resume=f'{PREENTRENO}/last.ckpt' if os.path.exists(f'{PREENTRENO}/last.ckpt') else None)

## 4. Entrenar con nuestras fotos

Los checkpoints van a Drive. Si Colab se corta, vuelve a correr las secciones 1, 2 y esta: **se reanuda sola**
desde `last.ckpt`. Con ~150 fotos, 40 épocas tardan del orden de 1 a 2 horas en una T4 (depende del día).

In [ ]:
from rfdetr import RFDETRNano

SALIDA = f'{DRIVE}/checkpoints/model_v1'
ultimo = f'{SALIDA}/last.ckpt'
pesos_externos = f'{PREENTRENO}/checkpoint_best_total.pth'

if os.path.exists(pesos_externos) and not os.path.exists(ultimo):
    modelo = RFDETRNano(pretrain_weights=pesos_externos)   # parte de lo aprendido con el dataset público
else:
    modelo = RFDETRNano()                                   # parte de los pesos de COCO

modelo.train(
    dataset_dir=str(DATASET),
    epochs=40,
    batch_size='auto',
    lr=1e-4,
    output_dir=SALIDA,
    resume=ultimo if os.path.exists(ultimo) else None,
)

## 5. Medir contra las metas (go/no-go)

| Métrica | Meta |
|---|---|
| mAP@0,5 por especie | ≥ 0,75 |
| Recall de chorito | ≥ 0,85 |
| Confusión chorito/cholga | ≤ 10 % |
| Error medio del conteo | ≤ 10 % del total |

In [ ]:
from rfdetr import RFDETR
mejor = RFDETR.from_checkpoint(f'{SALIDA}/checkpoint_best_total.pth')
metricas = mejor.evaluate(dataset_dir=str(DATASET), split='test')
print({k: round(float(v), 3) for k, v in metricas.items() if 'mAP' in k or 'mAR' in k})

In [ ]:
# Conteo, recall y confusión entre especies en el conjunto de prueba
import numpy as np
from PIL import Image

coco = json.load(open(DATASET / 'test' / '_annotations.coco.json'))
nombre = {c['id']: c['name'] for c in coco['categories']}
por_imagen = {}
for a in coco['annotations']:
    por_imagen.setdefault(a['image_id'], []).append(a)

def iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1]); x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter + 1e-9)

UMBRAL = 0.40
errores_pct, encontrados, reales, confusiones, emparejados = [], 0, 0, 0, 0
for im in coco['images']:
    reales_img = [a for a in por_imagen.get(im['id'], []) if nombre[a['category_id']] in ('chorito', 'cholga')]
    gt = [([a['bbox'][0], a['bbox'][1], a['bbox'][0] + a['bbox'][2], a['bbox'][1] + a['bbox'][3]], nombre[a['category_id']]) for a in reales_img]
    det = mejor.predict(Image.open(DATASET / 'test' / im['file_name']).convert('RGB'), threshold=UMBRAL)
    pred = [(list(b), CLASES[int(c)] if int(c) < len(CLASES) else str(c)) for b, c in zip(det.xyxy, det.class_id)]
    pred = [p for p in pred if p[1] in ('chorito', 'cholga')]
    if gt:
        errores_pct.append(abs(len(pred) - len(gt)) / len(gt) * 100)
    usados = set()
    for caja, especie in gt:
        if especie == 'chorito':
            reales += 1
        mejor_j, mejor_iou = None, 0.5
        for j, (pc, pe) in enumerate(pred):
            if j not in usados and iou(caja, pc) >= mejor_iou:
                mejor_j, mejor_iou = j, iou(caja, pc)
        if mejor_j is not None:
            usados.add(mejor_j); emparejados += 1
            if pred[mejor_j][1] != especie:
                confusiones += 1
            elif especie == 'chorito':
                encontrados += 1

print(f'Error medio del conteo: {np.mean(errores_pct):.1f} %   (meta ≤ 10 %)')
print(f'Recall de chorito:      {encontrados / max(reales, 1) * 100:.1f} %   (meta ≥ 85 %)')
print(f'Confusión de especie:   {confusiones / max(emparejados, 1) * 100:.1f} %   (meta ≤ 10 %)')
print('Nota: si la clase aparece con otro índice, revisa CLASES contra training_config.json de la carpeta de salida.')

## 6. Exportar a `.tflite` para la app

La exportación a TFLite de RF-DETR es **experimental** y sus dependencias chocan con las de entrenamiento.
Hazlo así:
1. Entorno de ejecución → **Desconectar y borrar el entorno de ejecución**.
2. Corre solo la celda 1 (Drive) y esta.

Genera FP32, FP16 e INT8 (rango dinámico). Se prueban los tres en la app; si INT8 pierde más de 3 puntos de mAP,
se usa FP16. Copia a Drive los `.tflite` **con su archivo `.rfdetr.json`** (lo necesita el cargador).

In [ ]:
!pip install -q "rfdetr[tflite]==1.12.0"
import tensorflow  # debe importarse ANTES que onnx, o la conversión puede quedarse pegada
from rfdetr import RFDETR

SALIDA = f'{DRIVE}/checkpoints/model_v1'
EXPORT = f'{DRIVE}/modelos/model_v1'
modelo = RFDETR.from_checkpoint(f'{SALIDA}/checkpoint_best_total.pth')
modelo.export(format='tflite', quantization='int8', output_dir=EXPORT)
print(os.listdir(EXPORT))

In [ ]:
# Archivos para la app (los recibe Ramón): labels.txt y modelo.json
import json
CLASES_APP = ['chorito', 'cholga', 'referencia']
open(f'{EXPORT}/labels.txt', 'w').write('\n'.join(CLASES_APP) + '\n')
json.dump({
    'version': 'model_v1',
    'arquitectura': 'RF-DETR Nano (Apache 2.0)',
    'archivo': 'model_v1.tflite',
    'entrada': {'formato': 'NHWC float32', 'normalizacion': 'ImageNet (media 0.485,0.456,0.406; desv. 0.229,0.224,0.225)'},
    'etiquetas': CLASES_APP,
    'umbral_confianza': 0.40,
    'referencia_mm': 40.0,
    'nota': 'Salidas sin nombre (output_0, output_1): cajas = tensor de rango 3 con última dimensión 4; '
            'puntajes = el otro tensor de rango 3. Confirmar el orden con el .rfdetr.json.',
}, open(f'{EXPORT}/modelo.json', 'w'), indent=2, ensure_ascii=False)
print(open(f'{EXPORT}/modelo.json').read())